# 01 — Data Audit and Initial Exploratory Analysis

This notebook performs the initial audit of the **Global Weather Repository** dataset used in the forecasting project.

The objective is to:

- validate data quality and schema consistency,
- inspect temporal and geographic coverage,
- identify the main forecasting target,
- verify the processed and engineered datasets,
- and screen candidate predictors before model training.

This notebook is focused on **data understanding and feature screening**.  
Model training and performance comparison should be documented in a separate notebook.

## 1. Environment setup and raw data loading

The first step is to load the raw dataset and confirm that the project can access the original CSV correctly.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RAW_PATH = "../data/raw/GlobalWeatherRepository.csv"
CLEAN_PATH = "../data/processed/GlobalWeatherRepository_processed.parquet"
FEATURE_PATH = "../data/processed/GlobalWeatherRepository_features.parquet"

df = pd.read_csv(RAW_PATH)
print("Raw dataset loaded successfully.")
print("Shape:", df.shape)

Raw dataset loaded successfully.
Shape: (130588, 41)


## 2. Schema inspection

This section checks dataset dimensionality, column availability, and variable types.  
This is important to confirm whether the dataset contains the temporal, geographic, meteorological, and environmental variables required by the project.

In [2]:
print("Shape of the dataset:", df.shape)
print("\nColumns in the dataset:")
print(df.columns.tolist())
print("\nData types of each column:")
print(df.dtypes)

Shape of the dataset: (130588, 41)

Columns in the dataset:
['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']

Data types of each column:
country                             str
location_name                       str
latitude                        float64
longitude                       float64
timezone                          

In [3]:
df.head()

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,wind_kph,wind_degree,wind_direction,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,8.3,13.3,338,NNW,1012.0,29.89,0.0,0.00,24,30,25.3,77.5,10.0,6.0,7.0,9.5,15.3,277.0,103.0,1.1,0.2,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,6.9,11.2,320,NW,1012.0,29.88,0.1,0.00,94,75,19.0,66.2,10.0,6.0,5.0,11.4,18.4,193.6,97.3,0.9,0.1,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,9.4,15.1,280,W,1011.0,29.85,0.0,0.00,29,0,24.6,76.4,10.0,6.0,5.0,13.9,22.3,540.7,12.2,65.1,13.4,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,7.4,11.9,215,SW,1007.0,29.75,0.3,0.01,61,100,3.8,38.9,2.0,1.0,2.0,8.5,13.7,170.2,64.4,1.6,0.2,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,8.1,13.0,150,SSE,1011.0,29.85,0.0,0.00,89,50,28.7,83.6,10.0,6.0,8.0,12.5,20.2,2964.0,19.0,72.7,31.5,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


## 3. Data quality checks

This section verifies whether the raw dataset contains missing values or duplicated rows.  
These checks help determine whether the cleaning stage requires aggressive remediation or only light preprocessing.

In [4]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

Series([], dtype: int64)

In [5]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0]

Series([], dtype: float64)

In [6]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


### Initial finding

The raw dataset shows **no missing values** and **no duplicated rows**.  
This indicates strong raw data quality and allows the preprocessing stage to focus mainly on type conversion, feature selection, and time-aware ordering instead of heavy imputation.

## 4. Temporal coverage

Because this project focuses on short-term weather forecasting, the temporal dimension is central.  
The `last_updated` field is converted to datetime format and inspected to verify coverage, consistency, and suitability for time-based modeling.

In [7]:
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

print("Min date:", df["last_updated"].min())
print("Max date:", df["last_updated"].max())
print("Unique timestamps:", df["last_updated"].nunique())

Min date: 2024-05-16 01:45:00
Max date: 2026-03-20 19:45:00
Unique timestamps: 21904


### Initial finding

The dataset spans from **May 2024 to March 2026** and includes a large number of unique timestamps.  
This confirms that the dataset is suitable for temporal analysis and next-step forecasting experiments.

## 5. Candidate target and domain-related variables

This section identifies the main temperature-related variables and maps additional meteorological and environmental predictors available in the dataset.

The goal is to define the main forecasting target and understand which explanatory variables can be tested later in the modeling stage.

In [8]:
[col for col in df.columns if "temp" in col.lower()]

['temperature_celsius', 'temperature_fahrenheit']

In [9]:
print("Temperature-related:", [col for col in df.columns if "temp" in col.lower()])
print("Precipitation-related:", [col for col in df.columns if "prec" in col.lower() or "rain" in col.lower()])
print("Air-quality-related:", [col for col in df.columns if "air" in col.lower() or "pm" in col.lower() or "ozone" in col.lower() or "no2" in col.lower()])
print("Wind-related:", [col for col in df.columns if "wind" in col.lower()])

Temperature-related: ['temperature_celsius', 'temperature_fahrenheit']
Precipitation-related: ['precip_mm', 'precip_in']
Air-quality-related: ['air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index']
Wind-related: ['wind_mph', 'wind_kph', 'wind_degree', 'wind_direction']


### Initial finding

The dataset contains both Celsius and Fahrenheit temperature variables, along with precipitation, wind, pressure, humidity, cloud cover, UV, and air-quality measurements.

For this project, **`temperature_celsius`** is used as the main target because it is directly interpretable and well aligned with short-term forecasting analysis.

## 6. Descriptive statistics and geographic coverage

This step explores the numeric range of the dataset and checks the distribution of observations across locations.  
The objective is to confirm that the project has broad spatial coverage and a rich set of continuous predictors.

In [10]:
df.describe(include="number").T

,count,mean,std,min,25%,50%,75%,max
latitude,130588.0,1.920416e+01,2.442009e+01,-4.130000e+01,3.870000e+00,1.725000e+01,4.040000e+01,6.415000e+01
longitude,130588.0,2.197448e+01,6.578836e+01,-1.752000e+02,-6.836100e+00,2.323610e+01,5.058000e+01,1.792200e+02
last_updated_epoch,130588.0,1.744908e+09,1.676659e+07,1.715849e+09,1.730452e+09,1.744968e+09,1.759388e+09,1.773989e+09
temperature_celsius,130588.0,2.138653e+01,9.692784e+00,-2.980000e+01,1.610000e+01,2.400000e+01,2.800000e+01,4.920000e+01
temperature_fahrenheit,130588.0,7.049754e+01,1.744686e+01,-2.160000e+01,6.100000e+01,7.520000e+01,8.240000e+01,1.206000e+02
wind_mph,130588.0,8.039797e+00,7.288469e+00,2.200000e+00,3.800000e+00,6.900000e+00,1.100000e+01,1.841200e+03
wind_kph,130588.0,1.294241e+01,1.172650e+01,3.600000e+00,6.100000e+00,1.120000e+01,1.760000e+01,2.963200e+03
wind_degree,130588.0,1.689087e+02,1.036019e+02,1.000000e+00,8.000000e+01,1.610000e+02,2.550000e+02,3.600000e+02
pressure_mb,130588.0,1.014097e+03,1.048143e+01,9.470000e+02,1.010000e+03,1.014000e+03,1.018000e+03,3.006000e+03
pressure_in,130588.0,2.994564e+01,3.094812e-01,2.796000e+01,2.983000e+01,2.993000e+01,3.006000e+01,8.877000e+01


In [ ]:
if "location_name" in df.columns:
    df["location_name"].value_counts().head(20)

### Initial finding

The dataset contains broad geographic coverage across multiple locations and a rich numeric feature space.  
This supports both global exploratory analysis and comparative forecasting experiments across cities.

## 7. Processed dataset validation

After the raw audit, the processed dataset is loaded to confirm that preprocessing preserved the expected structure and key variables.

In [11]:
clean_df = pd.read_parquet(CLEAN_PATH)
print("Processed dataset shape:", clean_df.shape)
clean_df.head()

Processed dataset shape: (130588, 32)


,country,location_name,latitude,longitude,timezone,last_updated,temperature_celsius,condition_text,wind_kph,wind_degree,pressure_mb,precip_mm,humidity,cloud,feels_like_celsius,visibility_km,uv_index,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,2024-05-31 16:15:00,16.0,Moderate rain,28.1,20,1014.0,0.29,82,50,16.0,10.0,4.0,36.3,186.9,87.3,2.7,1.3,2.2,3.4,1,1,05:36 AM,09:52 PM,02:59 AM,02:09 PM,Waning Crescent,47
1,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,2024-06-01 16:30:00,16.0,Overcast,15.1,330,1019.0,0.01,88,100,16.0,10.0,4.0,38.7,186.9,88.7,2.2,1.8,6.5,19.6,1,1,05:35 AM,09:53 PM,03:12 AM,03:34 PM,Waning Crescent,35
2,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,2024-06-04 16:15:00,18.0,Partly cloudy,25.9,270,1011.0,0.06,77,25,18.0,10.0,4.0,33.1,185.3,83.0,5.1,2.2,5.7,6.0,1,1,05:33 AM,09:56 PM,03:56 AM,07:57 PM,Waning Crescent,8
3,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,2024-06-05 16:15:00,15.0,Partly cloudy,11.2,330,1015.0,0.00,44,25,13.7,10.0,4.0,23.6,186.9,94.4,1.9,1.4,1.2,2.4,1,1,05:33 AM,09:56 PM,04:17 AM,09:25 PM,Waning Crescent,3
4,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,2024-06-11 16:15:00,15.2,Partly cloudy,24.1,280,1017.0,0.41,59,75,15.2,10.0,3.0,28.2,178.6,82.3,1.4,0.9,0.5,0.9,1,1,05:30 AM,10:01 PM,10:16 AM,01:31 AM,Waxing Crescent,21


### Initial finding

The processed dataset preserves the core temporal, geographic, meteorological, and environmental variables required for downstream feature engineering and modeling.

## 8. Feature engineering validation

This section validates the engineered dataset used in forecasting.  
It checks whether lagged temperatures, rolling statistics, and the future target variable were generated correctly.

In [12]:
feature_df = pd.read_parquet(FEATURE_PATH)
print("Feature dataset shape:", feature_df.shape)
feature_df[[
    "location_name",
    "last_updated",
    "temperature_celsius",
    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_roll_mean_3",
    "temp_roll_std_3",
    "target_temperature_next",
]].head(10)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/weather_features.parquet'

In [ ]:
feature_df[[
    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_roll_mean_3",
    "temp_roll_std_3",
    "target_temperature_next",
]].isna().sum()

### Initial finding

Missing values in lagged and rolling features are expected because the earliest records in each location do not have sufficient history, and the last record in each location cannot generate a future target.

These values are structural consequences of time-series feature engineering rather than raw data quality issues.

## 9. Correlation screening with the forecasting target

This section measures the linear association between numeric predictors and the next-step temperature target.  
The objective is not to perform final feature selection, but to identify strong signals, redundancy patterns, and promising feature groups for supervised learning.

In [ ]:
numeric_df = feature_df.select_dtypes(include=["number"])

corr_with_target = (
    numeric_df.corr(numeric_only=True)["target_temperature_next"]
    .sort_values(ascending=False)
)

corr_with_target

In [ ]:
corr_with_target.head(20)

In [ ]:
corr_with_target.tail(20)

In [ ]:
important_cols = [
    "target_temperature_next",
    "temperature_celsius",
    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_roll_mean_3",
    "temp_roll_std_3",
    "feels_like_celsius",
    "humidity",
    "cloud",
    "pressure_mb",
    "precip_mm",
    "wind_kph",
    "gust_kph",
    "visibility_km",
    "uv_index",
    "air_quality_PM2.5",
    "air_quality_PM10",
    "air_quality_Ozone",
    "latitude",
    "longitude",
]

corr_matrix = feature_df[important_cols].corr(numeric_only=True)
corr_matrix

In [ ]:
plt.figure(figsize=(14, 9))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Key Weather Features")
plt.tight_layout()
plt.show()

### Initial finding

The strongest associations with the target come from **current temperature**, **lagged temperatures**, **rolling temperature averages**, and **feels-like temperature**.  
This suggests that short-term forecasting is primarily driven by recent thermal history.

The heatmap also reveals substantial multicollinearity among temperature-related variables, which supports a staged modeling strategy based on structured feature blocks rather than indiscriminately using every variable at once.

It is also important to note that **correlation is used here only as an initial screening tool**.  
Final feature relevance should be evaluated through model-based importance and holdout performance.

## 10. Audit summary and transition to modeling

The dataset is well-structured, temporally valid, and suitable for short-term temperature forecasting.  
The raw data contains no missing values or duplicated rows, and the engineered feature set captures meaningful temporal dynamics.

The audit also indicates that:
- recent thermal history is the main driver of prediction,
- meteorological variables may add incremental value,
- geographic variables are likely useful in a global dataset,
- and some feature groups are clearly redundant.

Based on these findings, the next stage of the project should move to **model training and comparison**, including:

1. baseline evaluation,
2. structured feature-block comparison,
3. hyperparameter tuning,
4. feature importance analysis,
5. and error analysis by location or region.

For clarity and presentation quality, those modeling results should be documented in a separate notebook such as **`02_modeling_results.ipynb`**.